### Python packages used in this code

In [ ]:
import importlib
import sys
utils_path = './'
data_path = './data'
if utils_path not in sys.path:
    sys.path.append(utils_path)

from MyPackages import *
from MyFunctions import *
from MyTorch import *
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True
MyDevice = set_device('')



%matplotlib inline
MySEED = 373


def fun_Reload(pkg):
    module = importlib.import_module(pkg)
    module = importlib.reload(module)
    attrs = {attr: getattr(module, attr) for attr in dir(module)}
    globals().update(attrs)
    
fun_Reload('MyPackages')
fun_Reload('MyFunctions')
fun_Reload('MyTorch')



print('--Platform--')
print('OS :', platform.platform())
print('--Version--')
print('python : ', sys.version)
print('numpy :', np.__version__)
print('pandas :', pd.__version__)
print('sklearn :', pd.__version__)

## Environments
--Platform--

OS : Linux-6.5.11-linuxkit-x86_64-with-glibc2.31

--Version--

python :  3.9.7 | packaged by conda-forge | (default, Sep 29 2021, 19:23:11) 

[GCC 9.4.0]

numpy : 1.26.4

pandas : 2.0.3

sklearn : 2.0.3

# 1. Tm

In [ ]:
df = pd.read_csv('./Result/Tm.csv')
prop_list = df['Property'].unique()
df_dict = {}
for prop in prop_list:

    prop_df = df.query('Property == @prop')
    size_list = prop_df['NumSourceSamples'].unique()

    prop_df_mean = pd.DataFrame()
    prop_df_std = pd.DataFrame()
    prop_df_low_Tm = []
    prop_df_upper_Tm = []

    prop_df_low_Tm_IMD = []
    prop_df_upper_Tm_IMD = []


    for size in size_list:
        prop_df_mean = pd.concat([prop_df_mean,
                                prop_df.query('NumSourceSamples == @size').mean(numeric_only=True).to_frame().T])
        prop_df_std = pd.concat([prop_df_std,
                                prop_df.query('NumSourceSamples == @size').std(numeric_only=True).to_frame().T])
        prop_df_low_Tm.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tm_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_Tm.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tm_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)

        prop_df_low_Tm_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tm_IMD_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_Tm_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tm_IMD_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)



    df_dict[prop] = {'mean' : prop_df_mean,
                       'std' : prop_df_std, 
                       'Tm_MAE_upper' : prop_df_upper_Tm[1:],
                       'Tm_MAE_low' : prop_df_low_Tm[1:],

                       'Tm_IMD_MAE_upper' : prop_df_upper_Tm_IMD[1:],
                       'Tm_IMD_MAE_low' : prop_df_low_Tm_IMD[1:],
}

In [ ]:
target = 'Tm_MAE'


sns.set_style("whitegrid", {'grid.linestyle': '--'})
sns.set_context("paper", 1.5, {"lines.linewidth": 4})
sns.set_palette("hls",n_colors=len(prop_list))
fig, ax = plt.subplots(figsize=(10, 5))
marker = (['o', 'v', '^', 'X'] * int(len(prop_list)/4))[:len(prop_list)]

for prop, m in zip(prop_list, marker):

    x = df_dict[prop]['mean']['NumSourceSamples'][1:]
    y = df_dict[prop]['mean'][target][1:]

    ax.plot(x, y, label=prop, marker=m, markersize=15)
    #ax.scatter(x, y, s=100)

ax.axhline(y=df_dict[prop]['mean'][target].iloc[0],
            color='black',
            lw=3,
            ls='--',
            alpha=0.6,
            label='Direct learning'
            )

ax.set_xscale('log')
ax.set_xlabel("Number of Source Data", size = "large")
ax.set_ylabel("MAE [℃]", size = "large")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0,), borderaxespad=0)
plt.savefig(f'./Plot/{target}_single.pdf')


In [ ]:
target = 'Tm_MAE'

sns.set_style("whitegrid", {'grid.linestyle': '--'})
sns.set_context("paper", 1.5, {"lines.linewidth": 4})

v=5
h=6
fig, axes = plt.subplots(v, h, figsize=(50, 25))


for i in range(v):
    for j in range(h):
        num = h * i + j
        prop = prop_list[num]
        
        x = df_dict[prop]['mean']['NumSourceSamples'][1:]
        y = df_dict[prop]['mean'][target][1:]


        axes[i][j].plot(x, y, label=prop, color='b', marker='o', markersize=10)
        axes[i][j].fill_between(
            x,
            df_dict[prop][target + '_low'],
            df_dict[prop][target + '_upper'], 
            color='b', alpha=.15, label='90% Confidece Interval')



        axes[i][j].axhline(y=df_dict[prop]['mean'][target].iloc[0],
                    color='black',
                    lw=3,
                    ls='--',
                    alpha=0.6,
                    label='Direct learning'
                    )

        axes[i][j].set_xscale('log')
        #ax.set_xlim(10, 1000)
        #ax.set_ylim(0, 1)
        axes[i][j].set_xlabel("Number of Source data", size = "large")
        axes[i][j].set_ylabel("MAE [℃]", size = "large")
        axes[i][j].set_title(prop, size = "large")

plt.subplots_adjust(wspace=0.3,hspace=0.3)
plt.savefig(f'./Plot/{target}_subplot.pdf')

# 2. Tiso

In [ ]:
df = pd.read_csv('./Result/Tiso.csv')

prop_list = df['Property'].unique()
df_dict = {}
for prop in prop_list:

    prop_df = df.query('Property == @prop')
    size_list = prop_df['NumSourceSamples'].unique()

    prop_df_mean = pd.DataFrame()
    prop_df_std = pd.DataFrame()
    prop_df_low_Tm = []
    prop_df_upper_Tm = []

    prop_df_low_Tm_IMD = []
    prop_df_upper_Tm_IMD = []


    for size in size_list:
        prop_df_mean = pd.concat([prop_df_mean,
                                prop_df.query('NumSourceSamples == @size').mean(numeric_only=True).to_frame().T])
        prop_df_std = pd.concat([prop_df_std,
                                prop_df.query('NumSourceSamples == @size').std(numeric_only=True).to_frame().T])
        prop_df_low_Tm.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tiso_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_Tm.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tiso_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)

        prop_df_low_Tm_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tiso_IMD_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_Tm_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['Tiso_IMD_MAE'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)



    df_dict[prop] = {'mean' : prop_df_mean,
                       'std' : prop_df_std, 
                       'Tiso_MAE_upper' : prop_df_upper_Tm[1:],
                       'Tiso_MAE_low' : prop_df_low_Tm[1:],

                       'Tiso_IMD_MAE_upper' : prop_df_upper_Tm_IMD[1:],
                       'Tiso_IMD_MAE_low' : prop_df_low_Tm_IMD[1:],
}

In [ ]:
target = 'Tiso_MAE'


sns.set_style("whitegrid", {'grid.linestyle': '--'})
sns.set_context("paper", 1.5, {"lines.linewidth": 4})
sns.set_palette("hls",n_colors=len(prop_list))
fig, ax = plt.subplots(figsize=(10, 5))
marker = (['o', 'v', '^', 'X'] * int(len(prop_list)/4))[:len(prop_list)]

for prop, m in zip(prop_list, marker):

    x = df_dict[prop]['mean']['NumSourceSamples'][1:]
    y = df_dict[prop]['mean'][target][1:]

    ax.plot(x, y, label=prop, marker=m, markersize=15)
    #ax.scatter(x, y, s=100)

ax.axhline(y=df_dict[prop]['mean'][target].iloc[0],
            color='black',
            lw=3,
            ls='--',
            alpha=0.6,
            label='Direct learning'
            )

ax.set_xscale('log')
ax.set_xlabel("Number of Source Data", size = "large")
ax.set_ylabel("MAE [℃]", size = "large")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0,), borderaxespad=0)
plt.savefig(f'./Plot/{target}_single.pdf')

In [ ]:
target = 'Tiso_MAE'

sns.set_style("whitegrid", {'grid.linestyle': '--'})
sns.set_context("paper", 1.5, {"lines.linewidth": 4})

v=5
h=6
fig, axes = plt.subplots(v, h, figsize=(50, 25))


for i in range(v):
    for j in range(h):
        num = h * i + j
        prop = prop_list[num]
        
        x = df_dict[prop]['mean']['NumSourceSamples'][1:]
        y = df_dict[prop]['mean'][target][1:]


        axes[i][j].plot(x, y, label=prop, color='b', marker='o', markersize=10)
        axes[i][j].fill_between(
            x,
            df_dict[prop][target + '_low'],
            df_dict[prop][target + '_upper'], 
            color='b', alpha=.15, label='90% Confidece Interval')



        axes[i][j].axhline(y=df_dict[prop]['mean'][target].iloc[0],
                    color='black',
                    lw=3,
                    ls='--',
                    alpha=0.6,
                    label='Direct learning'
                    )

        axes[i][j].set_xscale('log')
        #ax.set_xlim(10, 1000)
        #ax.set_ylim(0, 1)
        axes[i][j].set_xlabel("Number of Source data", size = "large")
        axes[i][j].set_ylabel("MAE [℃]", size = "large")
        axes[i][j].set_title(prop, size = "large")

plt.subplots_adjust(wspace=0.3,hspace=0.3)
plt.savefig(f'./Plot/{target}_subplot.pdf')

# 3. Classification

In [ ]:
df = pd.read_csv('./Result/Class.csv', index_col=0)
prop_list = df['Property'].unique()
df_dict = {}
for prop in prop_list:

    prop_df = df.query('Property == @prop')
    size_list = prop_df['NumSourceSamples'].unique()

    prop_df_mean = pd.DataFrame()
    prop_df_std = pd.DataFrame()
    prop_df_low_f1 = []
    prop_df_upper_f1 = []
    prop_df_low_f1_IMD = []
    prop_df_upper_f1_IMD = []
    prop_df_low_N = []
    prop_df_upper_N = []
    prop_df_low_N_IMD = []
    prop_df_upper_N_IMD = []

    for size in size_list:
        prop_df_mean = pd.concat([prop_df_mean,
                                prop_df.query('NumSourceSamples == @size').mean(numeric_only=True).to_frame().T])
        prop_df_std = pd.concat([prop_df_std,
                                prop_df.query('NumSourceSamples == @size').std(numeric_only=True).to_frame().T])
        prop_df_low_f1.append(bootstrap((prop_df.query('NumSourceSamples == @size')['macro_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_f1.append(bootstrap((prop_df.query('NumSourceSamples == @size')['macro_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)
        prop_df_low_f1_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['IMD_macro_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_f1_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['IMD_macro_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)
        prop_df_low_N.append(bootstrap((prop_df.query('NumSourceSamples == @size')['N_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_N.append(bootstrap((prop_df.query('NumSourceSamples == @size')['N_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)
        prop_df_low_N_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['IMD_N_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.low)
        prop_df_upper_N_IMD.append(bootstrap((prop_df.query('NumSourceSamples == @size')['IMD_N_f1-score'].values,), np.mean, confidence_level=0.9, random_state=884).confidence_interval.high)


    df_dict[prop] = {'mean' : prop_df_mean,
                       'std' : prop_df_std, 
                       'macro_f1-score_upper' : prop_df_upper_f1[1:],
                       'macro_f1-score_low' : prop_df_low_f1[1:],
                       'IMD_macro_f1-score_upper' : prop_df_upper_f1_IMD[1:],
                       'IMD_macro_f1-score_low' : prop_df_low_f1_IMD[1:],
                        'N_f1-score_upper' : prop_df_upper_N[1:],
                       'N_f1-score_low' : prop_df_low_N[1:],
                       'IMD_N_f1-score_upper' : prop_df_upper_N_IMD[1:],
                       'IMD_N_f1-score_low' : prop_df_low_N_IMD[1:],
}


In [ ]:
target = 'N_f1-score'

sns.set_style("whitegrid", {'grid.linestyle': '--'})
sns.set_context("paper", 1.5, {"lines.linewidth": 4})
sns.set_palette("hls",n_colors=len(prop_list))
fig, ax = plt.subplots(figsize=(10, 5))
marker = (['o', 'v', '^', 'X'] * int(len(prop_list)/4))[:len(prop_list)]

for prop, m in zip(prop_list, marker):

    x = df_dict[prop]['mean']['NumSourceSamples'][1:]
    y = df_dict[prop]['mean'][target][1:]

    ax.plot(x, y, label=prop, marker=m, markersize=15)
    ax.scatter(x, y, s=100)

ax.axhline(y=df_dict[prop_list[0]]['mean'][target].to_list()[0],
            color='black',
            lw=3,
            ls='--',
            alpha=0.6,
            label='Direct learning'
            )


ax.set_xscale('log')
ax.set_xlabel("Number of Source Data", size = "large")
ax.set_ylabel("F1-score", size = "large")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0,), borderaxespad=0)
plt.savefig(f'./Plot/{target}_single.pdf')

In [ ]:
target = 'N_f1-score'

sns.set_style("whitegrid", {'grid.linestyle': '--'})
sns.set_context("paper", 1.5, {"lines.linewidth": 4})

v=5
h=6
fig, axes = plt.subplots(v, h, figsize=(50, 25))


for i in range(v):
    for j in range(h):
        num = h * i + j
        prop = prop_list[num]
        
        x = df_dict[prop]['mean']['NumSourceSamples'][1:]
        y = df_dict[prop]['mean'][target][1:]


        axes[i][j].plot(x, y, label=prop, color='b', marker='o', markersize=10)
 
        axes[i][j].fill_between(
            x,
            df_dict[prop][target + '_low'],
            df_dict[prop][target + '_upper'], 
            color='b', alpha=.15, label='90% Confidece Interval')



        axes[i][j].axhline(y=df_dict[prop_list[1]]['mean'][target].iloc[0],
                    color='black',
                    lw=3,
                    ls='--',
                    alpha=0.6,
                    label='Direct learning'
                    )

        axes[i][j].set_xscale('log')
        axes[i][j].set_xlabel("Number of Source data", size = "large")
        axes[i][j].set_ylabel("F1-score", size = "large")
        axes[i][j].set_title(prop, size = "large")

plt.subplots_adjust(wspace=0.3,hspace=0.3)
plt.savefig(f'./Plot/{target}_subplot.pdf')